In [61]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [62]:
df = pd.read_csv("power_dataset.csv")



In [63]:

df.head()

,node_id,timestamp,voltage,frequency,active_power,reactive_power,load_demand,renewable_output,temperature,humidity,energy_price,power_loss,region_id,stability_index,fault_indicator,attack_label
0,7,2024-01-01 00:00:00,1.028338,50.237769,307.517387,17.897299,508.356414,179.224063,43.626140,57.414603,12.284398,1.648733,1,15.518346,0,0
1,15,2024-01-01 00:01:00,1.053432,50.169283,361.815640,168.500907,393.022613,281.283786,38.587320,79.678709,12.867102,9.830852,3,15.494397,0,1
2,11,2024-01-01 00:02:00,0.945917,50.443237,241.606054,49.128098,213.454317,223.428783,22.136379,80.592591,10.429979,1.663154,3,15.415086,0,0
3,8,2024-01-01 00:03:00,0.924887,50.000674,402.337623,125.638795,319.933523,250.153680,38.039710,51.173218,9.248422,7.629393,1,15.485418,0,0
4,7,2024-01-01 00:04:00,1.007855,49.911674,466.410157,109.596754,462.529359,127.042484,26.576337,62.849441,9.008734,12.087418,4,15.492712,0,0


In [64]:
# Rename columns if needed (IMPORTANT - adjust to your dataset)
df = df.rename(columns={
    'voltage': 'voltage',
    'frequency': 'frequency',
    'active_power': 'active_power',
    'reactive_power': 'reactive_power',
    'attack_label': 'attack_label'
})

# --- Diff ---
import numpy as np

# --- Diff ---
df['freq_diff'] = df['frequency'].diff()
df['volt_diff'] = df['voltage'].diff()
df['p_diff'] = df['active_power'].diff()
df['q_diff'] = df['reactive_power'].diff()

# --- Rolling ---
df['freq_std_10'] = df['frequency'].rolling(10).std()

# --- Frequency deviation ---
df['freq_dev'] = abs(df['frequency'] - 50)

# --- Power relationships ---
df['S'] = np.sqrt(df['active_power']**2 + df['reactive_power']**2)
df['power_factor'] = df['active_power'] / (df['S'] + 1e-6)

df['pq_corr'] = df['active_power'].rolling(10).corr(df['reactive_power'])
df['v_p_ratio'] = df['voltage'] / (df['active_power'] + 1e-6)

# Drop NaN
df = df.dropna()

In [65]:
features = [
    'voltage', 'frequency',
    'active_power', 'reactive_power',

    'freq_diff', 'volt_diff',
    'p_diff', 'q_diff',

    'freq_std_10',
    'freq_dev',

    'power_factor'
]

X = df[features].values
y = df['attack_label'].values   

In [66]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [67]:
def create_sequences(X, y, seq_len=10):
    X_seq, y_seq = [], []
    
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len])
    
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y, seq_len=25)

In [68]:
split = int(0.8 * len(X_seq))

X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

In [69]:
class_weights = {0:1, 1:5}

In [70]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout



from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

model = Sequential([
    GRU(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),

    GRU(32),
    Dropout(0.3),

    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [71]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['Recall']
)

In [72]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=32,
    validation_data=(X_test, y_test),
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/40
500/500 [==============================] - 7s 11ms/step - loss: 0.9291 - recall: 0.0295 - val_loss: 0.5180 - val_recall: 0.0000e+00
Epoch 2/40
500/500 [==============================] - 5s 9ms/step - loss: 0.9255 - recall: 0.0055 - val_loss: 0.5370 - val_recall: 0.0000e+00
Epoch 3/40
500/500 [==============================] - 4s 9ms/step - loss: 0.9226 - recall: 6.1463e-04 - val_loss: 0.5047 - val_recall: 0.0000e+00
Epoch 4/40
500/500 [==============================] - 5s 9ms/step - loss: 0.9214 - recall: 0.0012 - val_loss: 0.4973 - val_recall: 0.0000e+00
Epoch 5/40
500/500 [==============================] - 5s 9ms/step - loss: 0.9179 - recall: 0.0000e+00 - val_loss: 0.4620 - val_recall: 0.0000e+00
Epoch 6/40
500/500 [==============================] - 5s 9ms/step - loss: 0.9177 - recall: 0.0018 - val_loss: 0.5061 - val_recall: 0.0000e+00
Epoch 7/40
500/500 [==============================] - 6s 12ms/step - loss: 0.9170 - recall: 0.0043 - val_loss: 0.5074 - val_recall: 0.0078


In [73]:
from sklearn.metrics import classification_report, confusion_matrix

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.3).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

125/125 [==============================] - 1s 3ms/step
[[1734 1875]
 [ 186  199]]
              precision    recall  f1-score   support

           0       0.90      0.48      0.63      3609
           1       0.10      0.52      0.16       385

    accuracy                           0.48      3994
   macro avg       0.50      0.50      0.39      3994
weighted avg       0.83      0.48      0.58      3994

